# 🔬 Laboratorio Analítico: Explotabilidad de SINDy

**Objetivo:** Este notebook analiza la base de datos `macro_backtest_db.csv` generada por el simulador `Walk-Forward`.
El propósito principal es separar el ruido de la señal y descubrir **Nichos Explotables** (combinaciones de Activo/Temporalidad) donde el modelo matemático de Nexus ofrece una ventaja estadística real para el Trading (Hit Ratio > 53% y Alpha > 0).

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Configurar pandas para visualizar el dataset ancho
pd.set_option('display.max_columns', None)

### 📉 Módulo 1: Limpieza y Tasa de Mortalidad Matemática
Evaluamos la robustez del modelo. ¿Con qué frecuencia la ecuación diferencial explota o SINDy no logra encontrar una solución válida?

In [ ]:
db_path = 'paper_kinetopus.csv'
try:
    df = pd.read_csv(db_path)
    print(f"📊 Base de Datos cargada: {len(df)} iteraciones totales.")
except FileNotFoundError:
    print(f"❌ ERROR: No se encontró el archivo {db_path}. Asegúrate de correr el Notebook 1 primero.")
    df = pd.DataFrame()

if not df.empty:
    # Crear columna de Escenario Combinado para segmentación fina
    df['Escenario'] = df['Periodo_Historia'] + ' (' + df['Intervalo_Velas'] + ')'
    
    # Calcular Tasa de Mortalidad por Activo y Escenario
    mortalidad = df.groupby(['Ticker', 'Escenario', 'Validez']).size().unstack(fill_value=0)
    
    if 'FALLO MATEMÁTICO' not in mortalidad.columns:
        mortalidad['FALLO MATEMÁTICO'] = 0
    if 'OK' not in mortalidad.columns:
        mortalidad['OK'] = 0
        
    mortalidad['Tasa de Fallo (%)'] = round((mortalidad['FALLO MATEMÁTICO'] / (mortalidad['OK'] + mortalidad['FALLO MATEMÁTICO'])) * 100, 2)
    display(mortalidad.sort_values('Tasa de Fallo (%)', ascending=False))
    
    # Depuración: A partir de aquí solo evaluamos predicciones válidas
    df_clean = df[df['Validez'] == 'OK'].copy()
    print(f"\n✅ Iteraciones Válidas (OK) listas para análisis predictivo: {len(df_clean)}")

📊 Base de Datos cargada: 1234 iteraciones totales.


,Validez,FALLO MATEMÁTICO,OK,Tasa de Fallo (%)
Ticker,Escenario,,,
MSFT,10y (1d),11,93,10.58
SPY,10y (1d),10,94,9.62
AAPL,10y (1d),4,100,3.85
MA,10y (1d),3,101,2.88
AMZN,10y (1d),3,101,2.88
ETH-USD,10y (1d),3,134,2.19
BAC,10y (1d),2,102,1.92
JPM,10y (1d),2,102,1.92
XLF,10y (1d),2,102,1.92



✅ Iteraciones Válidas (OK) listas para análisis predictivo: 1191


### 🎯 Módulo 2: La Prueba del Ácido (Hit Ratio B1 vs Azar)
Medimos si la predicción de la tendencia direccional en el **Tramo 1 (las primeras 10 velas del futuro)** logra vencer la barrera del azar (50%) de manera consistente.

In [4]:
if not df_clean.empty:
    # Agrupamos el Hit Ratio Promedio del Bloque 1
    edge_df = df_clean.groupby(['Ticker', 'Escenario'])['Hit_B1'].mean().reset_index()
    edge_df['Hit_B1 (%)'] = edge_df['Hit_B1'] * 100
    edge_df = edge_df.sort_values('Hit_B1 (%)', ascending=False)
    
    # Gráfico Interactivo de Barras
    fig = px.bar(edge_df, x='Ticker', y='Hit_B1 (%)', color='Escenario', barmode='group',
                 title="🎯 Hit Ratio Direccional a Corto Plazo (Tramo 1) vs Azar",
                 text_auto='.2f')
    
    # Líneas de referencia Críticas
    fig.add_hline(y=50, line_dash="dash", line_color="red", annotation_text="Punto de Quiebre Estadístico (50%)")
    fig.add_hline(y=53, line_dash="dot", line_color="#00ffcc", annotation_text="Umbral Explotable Recomendado (53%)")
    
    fig.update_layout(yaxis_range=[40, 100], template='plotly_dark')
    fig.show()

### ⚖️ Módulo 2.5: Calibración de Tasa Base (Base Rate Fallacy) y Coeficiente de Matthews (MCC)

En el análisis cuantitativo de trading, evaluar la precisión (*Hit Ratio*) de forma aislada puede conducir a graves falsos positivos debido a la **Paradoja de la Tasa Base (Base Rate Fallacy)**:

1. **El Sesgo de la Tasa Base (Prior Bias/Drift)**:
   Si un activo sube el 65% de las veces en una temporalidad dada debido a un fuerte sesgo alcista secular (drift), un predictor aleatorio "siempre alcista" obtendrá un **65% de Hit Ratio**. Esto parece fantástico, pero su ventaja predictiva real es de **0%** ya que no está aportando valor sobre la pura inercia del mercado.
   - Definimos la Tasa Base Alcista como: $P(Up) = \frac{1}{N}\sum \mathbb{I}(y_j = 1)$.
   - El **Benchmark del Azar Sesgado** es: $Benchmark = \max(P(Up), 1 - P(Up))$.
   - El **True Directional Alpha (TDA)** neto sobre la inercia del mercado es: $TDA = Hit\_Ratio - Benchmark$. Requerimos estrictamente que $TDA > 0\%$ para validar una ventaja real.

2. **Matthews Correlation Coefficient (MCC)**:
   Para abordar clases desbalanceadas (sesgos direccionales), el coeficiente de Matthews es el estándar de oro en machine learning. Evalúa de forma equilibrada las cuatro esquinas de la matriz de confusión (Verdaderos Positivos, Verdaderos Negativos, Falsos Positivos y Falsos Negativos):
   $$MCC = \frac{TP \times TN - FP \times FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$
   - **Rango**: $[-1, +1]$, donde $+1$ representa una predicción perfecta, $-1$ desacuerdo total y $0$ es equivalente al azar.
   - Dado que las direcciones de los precios en las ventanas son binarias ($y_j \in \{-1, 1\}$) y el acierto se define como $Hit_j = 1$ si $y_j = \hat{y}_j$ y $0$ de lo contrario, reconstruimos vectorialmente las predicciones direccionales de SINDy mediante:
     $$\hat{y}_j = (2 \cdot Hit_j - 1) \cdot y_j$$
     Esto nos permite calcular el MCC algebraicamente equivalente al coeficiente de correlación de Pearson:
     `MCC = np.corrcoef(y_true, y_pred)[0, 1]`
   - Establecemos un filtro de robustez estadística: $MCC > 0.05$ para clasificar una señal como verdaderamente explotable.

In [5]:
if not df_clean.empty:
    import numpy as np
    import pandas as pd
    import plotly.express as px
    import sys
    import os
    # Asegurar el path del proyecto para evitar ModuleNotFoundError: No module named 'src'
    sys.path.append(os.path.abspath('.'))
    sys.path.append(os.path.abspath('..'))

    from src.ui.market_loader import MarketLoader
    
    print("⌛ Iniciando calibración dinámica de tasa base con datos históricos del mercado...")
    
    # 1. Pipeline de reconstrucción direccional
    hit_cols = [c for c in df_clean.columns if c.startswith('Hit_B')]
    num_blocks = len(hit_cols)
    
    df_calib = df_clean.copy()
    for b in range(1, num_blocks + 1):
        df_calib[f'Act_Dir_B{b}'] = np.nan
        df_calib[f'Pred_Dir_B{b}'] = np.nan
        
    groups = df_calib.groupby(['Ticker', 'Periodo_Historia', 'Intervalo_Velas'])
    downloaded_data = {}
    
    for (ticker, period, interval), group_df in groups:
        key = (ticker, period, interval)
        if key not in downloaded_data:
            try:
                df_hist = MarketLoader.load_ticker_data(ticker, period=period, interval=interval)
                df_hist.columns = [str(c).title() for c in df_hist.columns]
                downloaded_data[key] = df_hist['Close'].values
            except Exception as e:
                print(f"⚠️ Error descargando histórico para {ticker} ({period} - {interval}): {e}")
                continue
                
        prices = downloaded_data[key]
        horizon = 150
        block_size = horizon // num_blocks
        
        for idx, row in group_df.iterrows():
            end_idx = int(row['Iteracion (Velas Vistas)'])
            if end_idx + horizon > len(prices):
                continue
                
            prices_sub = prices[end_idx : end_idx + horizon]
            
            # Bloque 1 (Tramo 1 - Velas 1 a 10)
            p_act_b1 = prices_sub[0 : block_size]
            hit_b1 = row['Hit_B1']
            
            # Dirección real local en B1
            delta_act_b1 = np.sign(p_act_b1[-1] - p_act_b1[0])
            if delta_act_b1 == 0: delta_act_b1 = 1.0
            
            # Predicción direccional reconstruida
            delta_pred_b1 = (2.0 * hit_b1 - 1.0) * delta_act_b1
            
            df_calib.at[idx, 'Act_Dir_B1'] = delta_act_b1
            df_calib.at[idx, 'Pred_Dir_B1'] = delta_pred_b1

    # Filtrar registros calibrados correctamente
    df_calibrated = df_calib.dropna(subset=['Act_Dir_B1']).copy()
    
    def compute_mcc(y_true, y_pred):
        if len(np.unique(y_true)) <= 1 or len(np.unique(y_pred)) <= 1:
            return 0.0
        corr = np.corrcoef(y_true, y_pred)[0, 1]
        return 0.0 if np.isnan(corr) else corr

    # Calcular estadísticas globales por escenario
    mcc_results = []
    grouped_calib = df_calibrated.groupby(['Ticker', 'Escenario'])
    
    for (ticker, escenario), group_df in grouped_calib:
        p_up_b1 = (group_df['Act_Dir_B1'] == 1.0).mean()
        benchmark_b1 = max(p_up_b1, 1.0 - p_up_b1)
        hit_b1 = group_df['Hit_B1'].mean()
        tda_b1 = hit_b1 - benchmark_b1
        mcc_b1 = compute_mcc(group_df['Act_Dir_B1'].values, group_df['Pred_Dir_B1'].values)
        
        mcc_results.append({
            'Ticker': ticker,
            'Escenario': escenario,
            'Prior_Up_B1 (%)': round(p_up_b1 * 100, 2),
            'Benchmark_B1 (%)': round(benchmark_b1 * 100, 2),
            'Hit_B1_Calib (%)': round(hit_b1 * 100, 2),
            'TDA_B1 (%)': round(tda_b1 * 100, 2),
            'MCC_B1': round(mcc_b1, 3)
        })
        
    df_calib_stats = pd.DataFrame(mcc_results)
    
    # Formatear visualmente para Plotly
    df_plot = df_calib_stats.melt(
        id_vars=['Ticker', 'Escenario', 'TDA_B1 (%)', 'MCC_B1'],
        value_vars=['Hit_B1_Calib (%)', 'Benchmark_B1 (%)'],
        var_name='Metrica', value_name='Porcentaje'
    )
    df_plot['Metrica'] = df_plot['Metrica'].replace({
        'Hit_B1_Calib (%)': 'Hit Ratio SINDy',
        'Benchmark_B1 (%)': 'Benchmark Azar (Tasa Base)'
    })
    
    # Gráfico interactivo emparejado
    fig = px.bar(
        df_plot, x='Ticker', y='Porcentaje', color='Metrica', barmode='group',
        facet_col='Escenario', facet_col_wrap=2,
        title="⚖️ Calibración de Tasa Base: Hit Ratio SINDy vs Benchmark Pasivo (Tramo 1)",
        text_auto='.1f',
        color_discrete_map={'Hit Ratio SINDy': '#00ffcc', 'Benchmark Azar (Tasa Base)': '#ff4d4d'}
    )
    fig.update_layout(yaxis_range=[30, 100], template='plotly_dark')
    fig.show()
    
    # Guardar en df_clean para ser utilizado por los módulos posteriores
    print(f"✅ Calibración finalizada con éxito. {len(df_calib_stats)} escenarios evaluados.")
    display(df_calib_stats.sort_values(by='MCC_B1', ascending=False).reset_index(drop=True))


⌛ Iniciando calibración dinámica de tasa base con datos históricos del mercado...


✅ Calibración finalizada con éxito. 11 escenarios evaluados.


,Ticker,Escenario,Prior_Up_B1 (%),Benchmark_B1 (%),Hit_B1_Calib (%),TDA_B1 (%),MCC_B1
0,ETH-USD,10y (1d),48.51,51.49,61.19,9.70,0.222
1,AMZN,10y (1d),52.48,52.48,58.42,5.94,0.175
2,C,10y (1d),47.57,52.43,56.31,3.88,0.122
3,SPY,10y (1d),46.81,53.19,53.19,0.00,0.065
4,MSFT,10y (1d),54.84,54.84,52.69,-2.15,0.049
5,JPM,10y (1d),50.98,50.98,51.96,0.98,0.042
6,AAPL,10y (1d),57.00,57.00,52.00,-5.00,0.038
7,BAC,10y (1d),53.92,53.92,50.98,-2.94,0.036
8,BTC-USD,10y (1d),51.57,51.57,51.57,0.00,0.029
9,XLF,10y (1d),50.00,50.00,50.98,0.98,0.020


### ⏳ Módulo 3: La Curva de Decaimiento Predictivo (The Decay Curve)
¿Hasta qué punto en el futuro la ecuación física pierde su fuerza de gravedad y es superada por el ruido del mercado? 
Aquí graficamos la caída de la precisión por **Tramos de 10 Velas** (del Tramo 1 al Tramo 15).

In [16]:
if not df_clean.empty:
    hit_cols = [f'Hit_B{i}' for i in range(1, 61)]
    
    # Promedio global según el Escenario (Periodo + Intervalo)
    decay_df = df_clean.groupby('Escenario')[hit_cols].mean().reset_index()
    
    # Transponer los datos (Melt) para que Plotly pueda dibujar la serie de tiempo
    decay_melted = pd.melt(decay_df, id_vars=['Escenario'], value_vars=hit_cols, 
                           var_name='Bloque_Futuro', value_name='Hit_Ratio')
    
    # Extraer el número de Tramo para el Eje X (1 al 15)
    decay_melted['Hit_Ratio'] = decay_melted['Hit_Ratio'] * 100
    decay_melted['Tramo_Futuro'] = decay_melted['Bloque_Futuro'].str.extract('(\d+)').astype(int)
    
    fig = px.line(decay_melted, x='Tramo_Futuro', y='Hit_Ratio', color='Escenario', markers=True,
                  title="⏳ Vida Útil de la Ecuación Física (Decaimiento Direccional por Tramos)",
                  labels={'Tramo_Futuro': 'Tramo Hacia el Futuro (Cada Tramo = 10 velas)', 'Hit_Ratio': 'Precisión Direccional (%)'})
    
    fig.add_hline(y=50, line_dash="dash", line_color="red", annotation_text="Zona de Ceguera (Ruido)")
    fig.update_xaxes(dtick=1) # Mostrar todos los números del 1 al 15 en el eje X
    fig.update_layout(template='plotly_dark')
    fig.show()

### 🗺️ Módulo 3.5: Hit Ratio Acumulado (Perspectiva Macro vs T=0)
A diferencia del Hit Local (que puede caer por reversión a la media), esta métrica responde a la pregunta: *¿El precio final del tramo N terminó en la misma dirección general (respecto al punto de partida $t=0$) que predijo SINDy?*

In [28]:
if not df_clean.empty and 'CumHit_B1' in df_clean.columns:
    cum_hit_cols = [f'CumHit_B{i}' for i in range(1, 61)]
    
    # Promedio global según el Escenario (Periodo + Intervalo)
    cum_decay_df = df_clean.groupby('Escenario')[cum_hit_cols].mean().reset_index()
    
    # Transponer los datos (Melt)
    cum_decay_melted = pd.melt(cum_decay_df, id_vars=['Escenario'], value_vars=cum_hit_cols, 
                           var_name='Bloque_Futuro', value_name='CumHit_Ratio')
    
    # Extraer el número de Tramo para el Eje X
    cum_decay_melted['CumHit_Ratio'] = cum_decay_melted['CumHit_Ratio'] * 100
    cum_decay_melted['Tramo_Futuro'] = cum_decay_melted['Bloque_Futuro'].str.extract(r'(\d+)').astype(int)
    
    fig = px.line(cum_decay_melted, x='Tramo_Futuro', y='CumHit_Ratio', color='Escenario', markers=True,
                  title="🗺️ Hit Ratio Acumulado (Destino Final vs Origen T=0)",
                  labels={'Tramo_Futuro': 'Tramo Hacia el Futuro (Cada Tramo = 5 velas)', 'CumHit_Ratio': 'Precisión Direccional Acumulada (%)'})
    
    fig.add_hline(y=50, line_dash="dash", line_color="red", annotation_text="Zona de Ceguera (Ruido)")
    fig.update_xaxes(dtick=1) # Mostrar todos los números del 1 al 15 en el eje X
    fig.update_layout(template='plotly_dark')
    fig.show()
elif not df_clean.empty:
    print("⚠️ La base de datos actual no contiene las columnas 'CumHit_B'. Por favor vuelve a generar la base de datos ejecutando el simulador (Notebook 1B).")


### 📈 Módulo 4: Propagación del Error Matemático (The Error Curve)
A diferencia del Hit Ratio (que evalúa dirección), aquí medimos el desvío absoluto porcentual (MAPE). 
¿A qué velocidad se degrada matemáticamente el valor exacto de la predicción a medida que viajamos hacia el futuro?
*(Nota: Utilizamos la **Mediana** en lugar de la Media, y un recorte visual en Y=100 para evitar que el gráfico colapse por iteraciones explosivas)*

In [20]:
if not df_clean.empty:
    # Verificamos si la DB nueva usa MAPE o si es la vieja con RMSE
    error_prefix = 'MAPE_B' if 'MAPE_B1' in df_clean.columns else 'RMSE_B'
    error_cols = [f'{error_prefix}{i}' for i in range(1, 61)]
    error_label = 'MAPE Típico (%)' if error_prefix == 'MAPE_B' else 'RMSE Típico Absoluto'
    
    # 1. Gráfico Segmentado por Escenario (MEDIANA en vez de MEDIA para escudo anti-outliers)
    error_df = df_clean.groupby('Escenario')[error_cols].median().reset_index()
    error_melted = pd.melt(error_df, id_vars=['Escenario'], value_vars=error_cols, 
                           var_name='Bloque_Futuro', value_name='Error_Promedio')
    error_melted['Tramo_Futuro'] = error_melted['Bloque_Futuro'].str.extract('(\d+)').astype(int)
    
    fig1 = px.line(error_melted, x='Tramo_Futuro', y='Error_Promedio', color='Escenario', markers=True,
                  title=f"📈 Propagación del Error por Tramos ({error_label}) - Segmentado (Mediana)",
                  labels={'Tramo_Futuro': 'Tramo Hacia el Futuro (Cada Tramo = 10 velas)', 'Error_Promedio': error_label})
    
    # Recorte Visual X=10, Y=100
    fig1.update_xaxes(dtick=1, range=[1, 10])
    fig1.update_layout(template='plotly_dark', yaxis_range=[0, 100])
    fig1.show()
    
    # 2. Gráfico Promedio Global (MEDIANA)
    global_error_df = df_clean[error_cols].median().reset_index()
    global_error_df.columns = ['Bloque_Futuro', 'Error_Promedio']
    global_error_df['Tramo_Futuro'] = global_error_df['Bloque_Futuro'].str.extract('(\d+)').astype(int)
    
    fig2 = px.area(global_error_df, x='Tramo_Futuro', y='Error_Promedio', markers=True,
                  title=f"🌍 Comportamiento Típico Global de Propagación del Error ({error_label})",
                  labels={'Tramo_Futuro': 'Tramo Hacia el Futuro (Cada Tramo = 10 velas)', 'Error_Promedio': error_label})
                  
    # Recorte Visual X=10, Y=100
    fig2.update_traces(line_color='#ff0066', fillcolor='rgba(255, 0, 102, 0.2)')
    fig2.update_xaxes(dtick=1, range=[1, 60])
    fig2.update_layout(template='plotly_dark', yaxis_range=[0, 100])
    fig2.show()

### 🔬 Módulo 4.5: Propagación del Error (Aciertos vs Fallos Direccionales)\n
Respondemos a la pregunta: *Cuando Kinetopus adivina la macrotendencia (CumHit = 1), ¿qué tan lejos está el precio exacto? ¿Y qué pasa cuando falla (CumHit = 0)?*\n
Esto separa la magnitud del error en dos grupos distintos.

In [21]:
if not df_clean.empty and 'CumHit_B1' in df_clean.columns:
    error_prefix = 'MAPE_B' if 'MAPE_B1' in df_clean.columns else 'RMSE_B'
    
    records = []
    
    # Extraemos el error condicionado al Hit Acumulado
    for idx, row in df_clean.iterrows():
        for i in range(1, 61):
            hit_col = f'CumHit_B{i}'
            error_col = f'{error_prefix}{i}'
            
            if hit_col in row and error_col in row:
                hit_val = row[hit_col]
                error_val = row[error_col]
                
                if pd.notna(hit_val) and pd.notna(error_val):
                    records.append({
                        'Tramo_Futuro': i,
                        'Resultado_Direccional': '✅ Acierto (CumHit=1)' if hit_val == 1.0 else '❌ Fallo (CumHit=0)',
                        'Error': error_val
                    })
                    
    if records:
        df_errors = pd.DataFrame(records)
        
        # Mediana del Error separada por Resultado Direccional
        median_errors = df_errors.groupby(['Tramo_Futuro', 'Resultado_Direccional'])['Error'].median().reset_index()
        
        fig = px.line(median_errors, x='Tramo_Futuro', y='Error', color='Resultado_Direccional', markers=True,
                      color_discrete_map={'✅ Acierto (CumHit=1)': '#00ffcc', '❌ Fallo (CumHit=0)': '#ff0066'},
                      title=f"🔬 Magnitud del Error ({error_prefix}): Cuando Acierta vs Cuando Falla la Tendencia",
                      labels={'Tramo_Futuro': 'Tramo Hacia el Futuro (10 velas/tramo)', 'Error': f'{error_prefix} Típico (Mediana %)'})
                      
        fig.update_xaxes(dtick=1, range=[1, 60]) # Nos enfocamos en los primeros 10 tramos
        fig.update_layout(template='plotly_dark', yaxis_range=[0, 100])
        fig.show()
elif not df_clean.empty:
    print("⚠️ Falta la columna CumHit_B1. Vuelve a ejecutar el simulador.")


### ⚔️ Módulo 5: El Edge Matemático (SINDy vs Naive Forecast)
Calculamos la ventaja (Alpha) que tiene SINDy sobre una predicción ciega (Línea Plana). Si Alpha > 0, SINDy ha vencido al ruido del mercado.

In [22]:
if not df_clean.empty and 'Naive_MAPE_B1' in df_clean.columns:
    # 1. Precalcular el Alpha Edge para todos los tramos disponibles (1 a 15)
    alpha_cols = []
    for i in range(1, 61):
        mape_col = f'MAPE_B{i}'
        naive_col = f'Naive_MAPE_B{i}'
        alpha_col = f'Alpha_B{i}'
        if naive_col in df_clean.columns and mape_col in df_clean.columns:
            # Alpha = Error Ingenuo - Error SINDy (Si es positivo, SINDy gana)
            df_clean[alpha_col] = df_clean[naive_col] - df_clean[mape_col]
            alpha_cols.append(alpha_col)
            
    if len(alpha_cols) > 0:
        # A. Gráfico de Decaimiento Segmentado por Escenario
        alpha_df = df_clean.groupby('Escenario')[alpha_cols].median().reset_index()
        alpha_melted = pd.melt(alpha_df, id_vars=['Escenario'], value_vars=alpha_cols, 
                               var_name='Bloque_Futuro', value_name='Alpha_Edge')
        alpha_melted['Tramo_Futuro'] = alpha_melted['Bloque_Futuro'].str.extract(r'(\d+)').astype(int)
        
        fig1 = px.line(alpha_melted, x='Tramo_Futuro', y='Alpha_Edge', color='Escenario', markers=True,
                      title="⚔️ Degradación del Alpha Matemático (SINDy vs Naive Forecast) - Por Escenario",
                      labels={'Tramo_Futuro': 'Tramo Hacia el Futuro (Cada Tramo = 10 velas)', 'Alpha_Edge': 'Ventaja Alpha (%)'})
        
        fig1.add_hline(y=0, line_dash="dash", line_color="white", annotation_text="La Roca Ciega (0% Ventaja)")
        fig1.update_xaxes(dtick=1)
        fig1.update_layout(template='plotly_dark')
        fig1.show()
        
        # B. Gráfico Promedio Global (MEDIANA de todo el Edge)
        global_alpha_df = df_clean[alpha_cols].median().reset_index()
        global_alpha_df.columns = ['Bloque_Futuro', 'Alpha_Edge']
        global_alpha_df['Tramo_Futuro'] = global_alpha_df['Bloque_Futuro'].str.extract(r'(\d+)').astype(int)
        
        fig2 = px.area(global_alpha_df, x='Tramo_Futuro', y='Alpha_Edge', markers=True,
                      title="🌍 Comportamiento Típico Global del Alpha Edge",
                      labels={'Tramo_Futuro': 'Tramo Hacia el Futuro (Cada Tramo = 10 velas)', 'Alpha_Edge': 'Ventaja Alpha Median Global (%)'})
                      
        fig2.update_traces(line_color='#00ffcc', fillcolor='rgba(0, 255, 204, 0.2)')
        fig2.add_hline(y=0, line_dash="dash", line_color="white", annotation_text="Pérdida de Edge")
        fig2.update_xaxes(dtick=1)
        fig2.update_layout(template='plotly_dark')
        fig2.show()
elif not df_clean.empty:
    print("⚠️ La base de datos no contiene las columnas de Naive Forecast. Vuelve a ejecutar el Backtest.")


### 🧠 Módulo 6: Filtro de Confianza (R² vs Realidad)
Si el Auto-Tuner dice que descubrió una física matemática casi perfecta (R² muy alto), ¿Esto se traduce en mayor dinero/acierto en el mundo real?

In [11]:
if not df_clean.empty:
    # Función para categorizar el R2 de SINDy en rangos legibles
    def categorize_r2(r2):
        if pd.isna(r2) or r2 < 0.3: return 'Bajo (<0.3)'
        elif r2 < 0.7: return 'Medio (0.3 - 0.7)'
        else: return 'Alto (>0.7)'
        
    df_clean['R2_Bucket'] = df_clean['SINDy R2'].apply(categorize_r2)
    
    # Agrupar y promediar
    r2_analysis = df_clean.groupby('R2_Bucket')['Hit_B1'].mean().reset_index()
    r2_analysis['Hit_B1 (%)'] = r2_analysis['Hit_B1'] * 100
    
    # Ordenar los rangos lógicamente de menor a mayor confianza
    order = ['Bajo (<0.3)', 'Medio (0.3 - 0.7)', 'Alto (>0.7)']
    r2_analysis['R2_Bucket'] = pd.Categorical(r2_analysis['R2_Bucket'], categories=order, ordered=True)
    r2_analysis = r2_analysis.sort_values('R2_Bucket')
    
    fig = px.bar(r2_analysis, x='R2_Bucket', y='Hit_B1 (%)', color='R2_Bucket',
                 title="🔮 Correlación entre el R² Teórico y la Precisión Direccional Real",
                 text_auto='.2f', color_discrete_sequence=['#ff4444', '#ffaa00', '#00ffcc'])
                 
    fig.add_hline(y=50, line_dash="dash", line_color="red")
    fig.update_layout(yaxis_range=[40, 100], template='plotly_dark')
    fig.show()

### 🏆 Módulo 7: Dashboard de Explotabilidad Cuantitativa
El reporte final para conectar el Trading Bot. Qué configuraciones (Ticker + Escenario) usar y cuáles desechar.

In [12]:
if not df_clean.empty:
    print("\n======================================================")
    print("🏆 DASHBOARD DE SEÑALES EXPLOTABLES (Nexus V1 - MCC Calibrated)")
    print("======================================================\n")
    
    error_col = 'MAPE_B1' if 'MAPE_B1' in df_clean.columns else 'RMSE_B1'
    error_name = 'MAPE_Típico (%)' if error_col == 'MAPE_B1' else 'RMSE_Típico'
    has_naive = 'Naive_MAPE_B1' in df_clean.columns
    
    # Diccionario de funciones de agregación para Named Aggregation
    agg_funcs = {
        'Total_Operaciones': ('Validez', 'count'),
        'Hit_B1': ('Hit_B1', 'mean'),  # B1 = Tramo 1 (Velas 1 a 10)
        'Hit_B3': ('Hit_B3', 'mean'),
        'CumHit_B3': ('CumHit_B3', 'mean') if 'CumHit_B3' in df_clean.columns else ('Hit_B3', 'mean'),  # B3 = Tramo 3
        error_col: (error_col, 'median'),
        'Drift_CUSUM_Medio': ('Drift (k)', 'mean')
    }
    if has_naive:
        agg_funcs['Naive_MAPE_B1'] = ('Naive_MAPE_B1', 'median')
        
    dashboard = df_clean.groupby(['Ticker', 'Escenario']).agg(**agg_funcs).reset_index()
    
    # Formateo visual
    dashboard['Hit_Ratio_CortoPlazo (%)'] = (dashboard['Hit_B1'] * 100).round(2)
    dashboard['Hit_Ratio_MedioPlazo (%)'] = (dashboard['Hit_B3'] * 100).round(2)
    if 'CumHit_B3' in dashboard.columns:
        dashboard['CumHit_Macro_MedioPlazo (%)'] = (dashboard['CumHit_B3'] * 100).round(2)
    dashboard[error_name] = dashboard[error_col].round(2)
    dashboard['Drift_CUSUM_Medio'] = dashboard['Drift_CUSUM_Medio'].round(2)
    
    if has_naive:
        dashboard['Alpha_Edge (%)'] = (dashboard['Naive_MAPE_B1'] - dashboard[error_col]).round(2)
        
    # Unir con la calibración del Módulo 2.5 (TDA y MCC)
    if 'df_calib_stats' in locals():
        dashboard = pd.merge(
            dashboard, 
            df_calib_stats[['Ticker', 'Escenario', 'Benchmark_B1 (%)', 'TDA_B1 (%)', 'MCC_B1']], 
            on=['Ticker', 'Escenario'], 
            how='left'
        )
    else:
        # Fallback si no se corrió el módulo 2.5
        dashboard['Benchmark_B1 (%)'] = 50.0
        dashboard['TDA_B1 (%)'] = dashboard['Hit_Ratio_CortoPlazo (%)'] - 50.0
        dashboard['MCC_B1'] = 0.0
        
    # Eliminamos las columnas base que ya no usamos
    cols_to_drop = ['Hit_B1', 'Hit_B3', error_col]
    if 'CumHit_B3' in dashboard.columns: cols_to_drop.append('CumHit_B3')
    if has_naive: cols_to_drop.append('Naive_MAPE_B1')
    dashboard.drop(columns=cols_to_drop, inplace=True)
    
    # Validación Híbrida de Explotabilidad Cuantitativa (Base Rate Bias Mitigation)
    condicion_mcc = (dashboard['TDA_B1 (%)'] > 0.0) & (dashboard['MCC_B1'] > 0.05)
    if has_naive:
        condicion = (dashboard['Hit_Ratio_CortoPlazo (%)'] >= 53.0) & (dashboard['Alpha_Edge (%)'] > 0) & condicion_mcc
    else:
        condicion = (dashboard['Hit_Ratio_CortoPlazo (%)'] >= 53.0) & condicion_mcc
        
    dashboard['Recomendación'] = np.where(
        condicion, 
        '🟩 APROBADO (Edge Real + MCC Robust)', 
        '🟥 RECHAZADO (Base Rate Bias / Ruido)'
    )
    
    # Ordenar y formatear visualmente
    dashboard = dashboard.sort_values('Hit_Ratio_CortoPlazo (%)', ascending=False).reset_index(drop=True)
    
    display(dashboard)
    
    # Resumen
    aprobados = len(dashboard[dashboard['Recomendación'].str.contains('APROBADO')])
    print(f"\n► Se han detectado {aprobados} configuraciones con Ventaja Estadística Real, TDA positivo y MCC robusto (>0.05).")



🏆 DASHBOARD DE SEÑALES EXPLOTABLES (Nexus V1 - MCC Calibrated)



,Ticker,Escenario,Total_Operaciones,Drift_CUSUM_Medio,Hit_Ratio_CortoPlazo (%),Hit_Ratio_MedioPlazo (%),CumHit_Macro_MedioPlazo (%),MAPE_Típico (%),Alpha_Edge (%),Benchmark_B1 (%),TDA_B1 (%),MCC_B1,Recomendación
0,ETH-USD,10y (1d),134,0.69,61.19,48.51,58.21,4.42,-0.86,51.49,9.70,0.222,🟥 RECHAZADO (Base Rate Bias / Ruido)
1,AMZN,10y (1d),101,0.79,58.42,44.55,50.50,1.97,0.11,52.48,5.94,0.175,🟩 APROBADO (Edge Real + MCC Robust)
2,C,10y (1d),103,1.10,56.31,53.40,50.49,2.03,-0.18,52.43,3.88,0.122,🟥 RECHAZADO (Base Rate Bias / Ruido)
3,SPY,10y (1d),94,1.40,53.19,53.19,57.45,0.96,-0.05,53.19,0.00,0.065,🟥 RECHAZADO (Base Rate Bias / Ruido)
4,MSFT,10y (1d),93,0.65,52.69,48.39,55.91,1.33,0.07,54.84,-2.15,0.049,🟥 RECHAZADO (Base Rate Bias / Ruido)
5,AAPL,10y (1d),100,0.72,52.00,42.00,45.00,1.86,-0.10,57.00,-5.00,0.038,🟥 RECHAZADO (Base Rate Bias / Ruido)
6,JPM,10y (1d),102,1.08,51.96,50.98,62.75,1.53,-0.03,50.98,0.98,0.042,🟥 RECHAZADO (Base Rate Bias / Ruido)
7,BTC-USD,10y (1d),159,1.80,51.57,53.46,54.09,3.46,-0.16,51.57,0.00,0.029,🟥 RECHAZADO (Base Rate Bias / Ruido)
8,BAC,10y (1d),102,1.18,50.98,50.98,54.90,1.77,0.13,53.92,-2.94,0.036,🟥 RECHAZADO (Base Rate Bias / Ruido)
9,XLF,10y (1d),102,1.34,50.98,53.92,58.82,1.22,0.00,50.00,0.98,0.020,🟥 RECHAZADO (Base Rate Bias / Ruido)



► Se han detectado 1 configuraciones con Ventaja Estadística Real, TDA positivo y MCC robusto (>0.05).


### 🥊 Módulo 6: Duelo de Titanes (SINDy vs Auto-ARIMA)\n
Aquí enfrentamos a la Física Computacional (Kinetopus) contra la Estadística Tradicional (Auto-ARIMA).\n
Cargamos ambas bases de datos simuladas bajo las mismas condiciones de aislamiento (Walk-Forward) y comparamos su Hit Ratio Acumulado global para ver en qué momento el modelo estadístico se rinde y se aplana, mientras que la física sigue la onda.

In [27]:
import os

db_sindy = 'backtest_kinetopus.csv'
db_arima = 'backtest_predictive.csv'

if os.path.exists(db_sindy) and os.path.exists(db_arima):
    df_s = pd.read_csv(db_sindy)
    df_a = pd.read_csv(db_arima)
    
    df_s = df_s[df_s['Validez'] == 'OK']
    df_a = df_a[df_a['Validez'] == 'OK']
    
    if not df_s.empty and not df_a.empty and 'CumHit_B1' in df_s.columns and 'CumHit_B1' in df_a.columns:
        cum_cols = [f'CumHit_B{i}' for i in range(1, 16)]
        
        # Promedio global de Hit Ratio por Tramo\n
        s_median = df_s[cum_cols].mean() * 100
        a_median = df_a[cum_cols].mean() * 100
        
        compare_df = pd.DataFrame({
            'Tramo_Futuro': range(1, 16),
            'SINDy_CumHit': s_median.values,
            'ARIMA_CumHit': a_median.values
        })
        
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=compare_df['Tramo_Futuro'], y=compare_df['SINDy_CumHit'],
                                 mode='lines+markers', name='Física (Kinetopus / SINDy)', line=dict(color='#00ffcc', width=3)))
        fig.add_trace(go.Scatter(x=compare_df['Tramo_Futuro'], y=compare_df['ARIMA_CumHit'],
                                 mode='lines+markers', name='Estadística (Auto-ARIMA)', line=dict(color='#ffaa00', width=3)))
        
        fig.update_layout(title="🥊 Comparativa de Supervivencia Direccional (Hit Ratio Acumulado Global)",
                          xaxis_title="Tramo Hacia el Futuro (10 velas c/u)",
                          yaxis_title="Precisión Acumulada (%)",
                          template='plotly_dark')
        fig.add_hline(y=50, line_dash="dash", line_color="white", annotation_text="Roca Ciega (50%)")
        fig.update_xaxes(dtick=1)
        fig.show()
    else:
        print("⚠️ Las bases de datos no tienen las columnas necesarias o están vacías.")
else:
    print("⚠️ Aún no has generado la base de datos de ARIMA. Ve al Notebook 1C y córrelo.")